# 03 – Files and formats

Texts reach us in different formats: as plaintext, as PDF, as XML, as JSON. The format decides what we can do with a text and how much work it takes to get there. In this session we read one theatre review from the Süddeutsche Zeitung of 1 February 1985 in three formats, and save it in a fourth

**Before we start**

1. Download the four files for this session from Moodle
2. Open [Google Drive](https://drive.google.com) and log in with the Google account you use for Colab
3. In "My Drive", create the folders below (the folder `review-course/data` may already be there from the last sessions) and upload each file into its folder:

```
My Drive/
└── review-course/
    └── data/
        └── intern/
            ├── SZ-1985-Feuilleton-txt/
            │   └── 1001-SZ-01021985.txt
            ├── PDF-samples/
            │   ├── SZ-01021985-Seite50.pdf
            │   └── SZ-01021985-Seite50-scan.pdf
            └── XML-samples/
                └── 1001-SZ-01021985.xml
```

The notebook expects the files exactly there. Please keep the folder names and the file names as they are

### 1.

Libraries are collections of code that others have written. With import we can use them

In [ ]:
# import makes a library available; its functions are called with the name of the library and a dot
import math
print(math.sqrt(16))

In [ ]:
# with from ... import we take a single thing out of a library and can use it without the library name
# pathlib helps with files and folders: a Path describes where a file or folder is
from pathlib import Path

In [ ]:
# a library can get a shorter name with as; ElementTree reads XML files
import xml.etree.ElementTree as ET

# json reads and writes JSON files
import json

math, pathlib, xml and json come with python, they are part of the so-called standard library. Other libraries have to be installed first. pypdf, which reads PDF files, is one of them

In [ ]:
# importing a library that is not installed is a typical mistake
import theaterkritik

In [ ]:
# a line starting with ! is not python but a command for the computer Colab runs on; pip installs libraries
!pip install pypdf

In [ ]:
# now the import works
from pypdf import PdfReader

### 2.

Our files are in Google Drive, in the folder `review-course/data/intern/`. To read them, we first connect the Drive

In [ ]:
# this library only exists in Colab; Google will ask for permission to access your Drive – allow it
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# the / joins the parts of a path, folder by folder; exists() checks whether the file is really there (a quick test against typos)
DATA_DIR = Path("/content/drive/MyDrive/review-course/data")
txt_path = DATA_DIR / "intern" / "SZ-1985-Feuilleton-txt" / "1001-SZ-01021985.txt"
print(txt_path)
print(txt_path.exists())

A plaintext file contains nothing but characters. This one holds the whole Feuilleton of the issue of 1 February 1985. On disk, though, a computer stores only numbers (bytes): an encoding is the table that says which number stands for which character. UTF-8 is today's standard and covers the letters of almost every writing system, ä, ö, ü and ß included; older files often use other encodings, and then the same bytes turn into different letters

In [ ]:
# read_text() reads a file as one long string; the encoding tells python how the letters are stored in the file
plain = txt_path.read_text(encoding="utf-8")
print(len(plain))
print(plain[:1000])

In [ ]:
# with find() and a slice we look for our review, a production of Shakespeare's "Was ihr wollt" in Hamburg
position = plain.find("Endspiel")
print(plain[position - 200:position + 600])

In [ ]:
# count() counts how often a string appears: the whole Feuilleton has hardly any line breaks ("\n")
print(plain.count("\n"))

Plaintext is easy to read, for us and for python: one line of code, and we have the whole text as a string. But look at `BFEndspiel: heiter, doch nicht sonnig„Was ihr wollt“ am Hamburger Thalia-TheaterUlyrien ist ...`: the initials of the previous author, the title, the subtitle and the first sentence run into each other. Where does an article begin, where does it end, who wrote it, on which page was it printed? The file does not say. Everything that is not a character is lost

### 3.

A PDF describes what a page looks like: which letter, line or picture sits where. It is made for printing and for the eye, not for further processing. Our PDF shows page 50 of the same issue, laid out like a newspaper page (we generated it ourselves from the archive)

In [ ]:
# a PDF is not plain text: read_bytes() reads the raw bytes, and they are mostly instructions for drawing the page
pdf_path = DATA_DIR / "intern" / "PDF-samples" / "SZ-01021985-Seite50.pdf"
raw = pdf_path.read_bytes()
print(raw[:600])

In [ ]:
# PdfReader understands the structure of the file; pages is a list of the pages
reader = PdfReader(pdf_path)
print(len(reader.pages))

In [ ]:
# extract_text() collects the letters of a page and tries to put them into the right order
page_text = reader.pages[0].extract_text()
print(page_text[:1500])

The text is there, but so is the layout: the running head and the page number come before the title, every line of the narrow column ends with a line break, and many words are hyphenated

In [ ]:
# a hyphenated word is cut into two words: we would not find "Wirrnis" when we count words
words = page_text.split()
print("Wirrnis" in words)
print("Wirr-" in words)

In [ ]:
# replace() swaps one string for another: we glue the hyphenated words back together and turn line breaks into spaces
repaired = page_text.replace("-\n", "").replace("\n", " ")
print("Wirrnis" in repaired.split())
print(repaired[:800])

In [ ]:
# but the repair has a price: a hyphen at the end of a line can also belong to the word itself
print("Pierrot-\nKostüm" in page_text)
print("PierrotKostüm" in repaired)

In [ ]:
# the review continues on the second page – and the running head reappears in the middle of the text
print(reader.pages[1].extract_text()[:400])

In [ ]:
# where does Burkhardt's review end and the next article begin? nothing in the text tells us
position = page_text.find("BURKHARDT")
print(page_text[position - 200:position + 200])

This PDF was born digital and still contains its letters. Many PDFs are scans of printed paper: they contain only pictures of the pages

In [ ]:
scan_path = DATA_DIR / "intern" / "PDF-samples" / "SZ-01021985-Seite50-scan.pdf"
scan = PdfReader(scan_path)
print(len(scan.pages))
# len() of the extracted text: how many characters did pypdf find?
print(len(scan.pages[0].extract_text()))

Nothing. To get the text out of a picture we need OCR (optical character recognition), a program that guesses letters from their shapes. The SZ archive has gone through this: the printed newspaper of 1985 was scanned and recognised. Traces of it remain in every format we read today, e.g. "Ulyrien" for "Illyrien" in the first sentence, or "Stimnarben" for "Stirnnarben" further down

| | plaintext | PDF |
|---|---|---|
| to read it we need | `read_text()` and the right encoding | a library like pypdf, for scans also OCR |
| what we get | the characters, nothing else | the characters plus the traces of the layout: line breaks, hyphens, running heads |
| advantages | simple, small, readable by any program, stable for decades | looks exactly like the original, good for reading and citing |
| disadvantages | the structure is lost: titles, authors, article boundaries | text order and words have to be repaired, a scan contains no text at all |

### 4.

The archive delivers the same issue in XML. XML is plaintext with tags: `<titel>` opens an element, `</titel>` closes it, and elements can contain other elements, like folders contain folders. The tags name what a piece of text *is*

In [ ]:
xml_path = DATA_DIR / "intern" / "XML-samples" / "1001-SZ-01021985.xml"

# this file uses an older encoding; reading it as utf-8 is a mistake, and python says so
xml_path.read_text(encoding="utf-8")

In [ ]:
# the first line of the file names its encoding
print(xml_path.read_text(encoding="iso-8859-1")[:1500])

Every `<artikel>` contains `<metadaten>` (data about the article) and `<inhalt>`; `<inhalt>` contains the `<titel-liste>` and the `<text>`, and the text consists of paragraphs, `<absatz>`. We do not have to cut this apart ourselves: ElementTree reads the tags for us

In [ ]:
# ET.parse() reads the file and understands its structure; the root is the outermost element
tree = ET.parse(xml_path)
root = tree.getroot()
print(root.tag)

In [ ]:
# findall() gives us a list of all elements with a certain tag
articles = root.findall("artikel")
print(len(articles))

In [ ]:
# findtext() gives us the text inside an element; the path leads through the elements like through folders
print(articles[0].findtext("inhalt/titel-liste/titel"))
print(articles[0].findtext("metadaten/quelle/datum"))

In [ ]:
# a loop finds our review by its title
for article in articles:
    if "Endspiel" in article.findtext("inhalt/titel-liste/titel"):
        review = article

print(review.findtext("inhalt/titel-liste/titel"))
print(review.findtext("inhalt/titel-liste/ressort"))
print(review.findtext("metadaten/quelle/seite-start"))

In [ ]:
# ET.tostring() shows an element with all its tags: here everything the archive knows about our review
print(ET.tostring(review.find("metadaten"), encoding="unicode"))

In [ ]:
# the text of an article consists of paragraphs (<absatz>)
paragraphs = review.findall("inhalt/text/absatz")
print(len(paragraphs))

In [ ]:
# some paragraphs contain further tags, like <b> for bold; itertext() collects all the text inside an element
# "".join() glues the pieces together into one string, and "\n" puts a line break after each paragraph
text = ""
for paragraph in paragraphs:
    text = text + "".join(paragraph.itertext()) + "\n"
print(text)

With XML the article boundaries, the title, the author, the page and the paragraphs are no longer guesswork: they are named. The price is that we have to know the tags (which the archive documents in a schema, here `pmg-artikel-liste.xsd`), that we need a parser like ElementTree, and that the files are long: most of what they contain is tags. And XML only stores what someone has put into it: the OCR errors are still there

### 5.

We now save the review in a format of our own. For this we need one more structure: a dictionary holds pairs of keys and values, and we look up a value by its key, like a word in a dictionary

In [ ]:
# a dictionary is written with curly brackets; each key is followed by a colon and its value
record = {
    "zeitung": "Süddeutsche Zeitung",
    "datum": review.findtext("metadaten/quelle/datum"),
    "titel": review.findtext("inhalt/titel-liste/titel"),
}
print(record)
print(record["titel"])

In [ ]:
# a new pair is added by assigning a value to a new key
record["text"] = text
print(record.keys())

JSON (JavaScript Object Notation) writes dictionaries and lists as text, in almost the same way python writes them. Almost every programming language can read it

In [ ]:
# json.dumps() turns the dictionary into a string; ensure_ascii=False keeps the umlauts, indent=2 puts every key on its own line
as_json = json.dumps(record, ensure_ascii=False, indent=2)
print(as_json[:800])

In [ ]:
# write_text() writes the string into a file in the Drive; careful: a file that already exists is overwritten
out = DATA_DIR / "intern" / "SZ-01021985-endspiel.json"
out.write_text(as_json, encoding="utf-8")
print("written to", out)

In [ ]:
# json.loads() turns the text of the file back into a dictionary, and we can use the keys again
loaded = json.loads(out.read_text(encoding="utf-8"))
print(type(loaded))
print(loaded["titel"])
print(loaded["text"][:200])

Like XML, JSON names its parts, but it is shorter and maps directly onto python's dictionaries and lists. XML is better at marking up running text (a `<b>` in the middle of a sentence), JSON is better at records: one entry, a few fields, a text

| | plaintext | PDF | XML | JSON |
|---|---|---|---|---|
| to read it we need | `read_text()` | pypdf (and OCR for scans) | ElementTree and knowledge of the tags | `json.loads()` |
| structure | none | layout only | named elements, nested | keys and values, lists |
| good for | reading, counting words | looking at the original | archives, editions, marked-up text | our own collections of data |
| problem | everything except the characters is lost | text has to be repaired | long, needs a schema | not made for running text with markup |

### Exercise

Our JSON file only contains newspaper, date, title and text. The XML knows more

1. Print the whole review with `ET.tostring(review, encoding="unicode")` and note which information we have not saved yet
2. Extend `record` with at least the author, the subtitle, the page, the section (`ressort`) and the article id
3. Add the number of words of the text (you know how to count them from the last session)
4. Write the date as `01.02.1985` instead of `01021985` (hint: slices)
5. Save the record as a new JSON file and read it back in to check

Bonus: `json.dumps()` also accepts a list of dictionaries. Save every Feuilleton article of the issue in one JSON file (leave out the articles whose `rubrik` is `Ganzseite` – they repeat the text of whole pages)

To think about: which information would we want for a theatre review that the archive does not give us? (The play, the theatre, the director, the date of the performance ...) Where could it come from?